# Naive Bayes

Naive Bayes is a probabilistic machine learning algorithm based on Bayes' Theorem.
Basically, it tells us the probability that the target belongs to a specific class, given a set of input features.

Bayes conditional probability theorem equation:

$$ P(y|X) = \frac{P(X|y) \cdot P(y)}{P(X)} $$

Where:
- **$P(y|X)$ (Posterior Probability)**: The probability of the target class $y$ given the input features $X$. This is what the model is trying to predict.
- **$P(X|y)$ (Likelihood)**: The probability of observing the input features $X$ given that the target is class $y$.
- **$P(y)$ (Prior Probability)**: The initial probability of the class $y$ occurring in the dataset, before looking at the features.
- **$P(X)$ (Evidence/Marginal Probability)**: The overall probability of the input features $X$ occurring across all classes.

It assumes that all of the given features are independent of each other, which means it ignores any correlation between the given features.

$P(X∣y)=P(x_1∣y)⋅P(x_2∣y)⋅...⋅P(x_n∣y)$

Due to its simplified nature and scalability on larger datasets, it is used in applications such as text classification in spam detection, sentiment analysis, etc.


# Types of Naive Bayes



# Gaussian Naive Bayes

Gaussian Naive Bayes is used when the features are continuous-valued.

Each feature is assumed to follow a normal distribution within each class, the model will learn the shape of the data for every class using:

- mean ($\mu$)  
- variance or standard deviation ($\sigma^2$ or $\sigma$)  

This lets the model estimate how likely a numerical value is for a given class.



### Likelihood Model

The probability of observing a feature value $x_i$ given a class $y$ is modeled as:

$$
P(x_i \mid y) = \frac{1}{\sqrt{2\pi \sigma_y^2}} \exp\left(-\frac{(x_i - \mu_y)^2}{2\sigma_y^2}\right)
$$


For each class, the model looks at the training data and calculates:

- the average value of the feature  
- how much the values spread out around that average  

Then, when a new input comes in, the model checks how close its feature value is to the class mean.

If the value is close to the mean, the probability is higher.  
If the value is far from the mean, the probability becomes lower.

Due to this the classifier is faster and easier to train.




In [10]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

gnb = GaussianNB()
gnb.fit(X_train, y_train)


y_pred = gnb.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9736842105263158

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.93      0.96        43
           1       0.96      1.00      0.98        71

    accuracy                           0.97       114
   macro avg       0.98      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114





## Multinomial Naive Bayes

Multinomial Naive Bayes is designed for discrete features, especially those representing counts or frequencies.

This model is widely used in text classification, where features correspond to word counts in a document.

#### Likelihood Model

$$
P(x \mid y) \propto \prod_{i=1}^{n} P(x_i \mid y)^{x_i}
$$


- $x_i$ represents the count of feature $i$ (e.g., number of times a word appears)  
- $P(x_i \mid y)$ represents the probability of that feature occurring in class $y$  

The model essentially evaluates how likely a document is generated by a class based on word frequencies.


#### Smoothing

To handle zero probabilities (when a word does not appear in training data), Laplace smoothing is applied:

$$
P(x_i \mid y) = \frac{\text{count}(x_i, y) + \alpha}{\sum_j \text{count}(x_j, y) + \alpha n}
$$

- $\alpha$ is a smoothing parameter  



#### Applications

- spam detection based on word frequency  
- sentiment analysis of reviews  
- document categorization  

#### Key Characteristics

- handles high-dimensional sparse data well  
- robust to irrelevant features  
- assumes feature counts are independent  



## Text Classification with Multinomial Naive Bayes


Here, Bayes' Theorem calculates the probability that an email is spam given the words it contains. It assumes each word's presence is independent of the others. We use `MultinomialNB` for this, which works well with discrete word counts.

First, we must convert our text into numbers using `CountVectorizer`, which creates a "Bag of Words" (a count of how many times each word appears in each message).

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

texts = [
    "Win a free iPhone now",
    "Call this number to claim your prize",
    "Hey, are we still meeting for lunch?",
    "Don't forget to submit the report by 5 PM",
    "Congratulations! You've been selected for a free gift",
    "Can you pick up some milk on your way home?"
]

labels = [1, 1, 0, 0, 1, 0] 

model = make_pipeline(CountVectorizer(), MultinomialNB())

model.fit(texts, labels)

test_texts = [
    "Free entry for a chance to win a prize!",
    "Let me know when you are available.",
    "Claim your free gift by clicking here"
]

predictions = model.predict(test_texts)


for text, pred in zip(test_texts, predictions):
    status = "SPAM" if pred == 1 else "NOT SPAM"
    print(f"[{status}] -> '{text}'")


[SPAM] -> 'Free entry for a chance to win a prize!'
[NOT SPAM] -> 'Let me know when you are available.'
[SPAM] -> 'Claim your free gift by clicking here'


## Bernoulli Naive Bayes

Bernoulli Naive Bayes is used when each feature in the dataset is binary.

The model only checks whether the feature is present or absent. This makes it especially useful when the presence of a feature carries more information than its frequency.

So the model focuses on whether important words exist in the input rather than how often they occur.

### Likelihood Model

For a binary feature, the probability of observing it under a class label is written as:

$$
P(x_i \mid y) = p_i^{x_i}(1-p_i)^{1-x_i}
$$


If $x_i = 1$, then:

$$
P(x_i \mid y) = p_i
$$

If $x_i = 0$, then:

$$
P(x_i \mid y) = 1 - p_i
$$

So for each feature, the model learns:

- how likely the feature is to be present in a given class  
- how likely the feature is to be absent in that class  

For example, in email classification, it can learn whether a word such as “free” or “urgent” is more likely to appear in spam emails than in normal emails. The final prediction is then based on which class makes the observed pattern of present and absent features most likely.

It is especially useful in tasks like text classification, keyword detection, and other presence/absence based problems.


In [4]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
data = fetch_20newsgroups(subset='all',
                         categories=['sci.space', 'rec.autos'],
                         remove=('headers', 'footers', 'quotes'))

X = data.data
y = data.target

# Convert text to binary features
vectorizer = CountVectorizer(binary=True)
X = vectorizer.fit_transform(X)

# Split
Xtr, Xtest, ytr, ytest = train_test_split(X, y, test_size=0.3, random_state=42)

# Model
model = BernoulliNB()
model.fit(Xtr, ytr)

# Predict
pred = model.predict(Xtest)

print("Accuracy:", accuracy_score(ytest, pred))
print(classification_report(ytest, pred))

print(f"Example prediction: Input:{data.data[0]} => Predicted: {model.predict(vectorizer.transform([data.data[0]]))[0]}")

Accuracy: 0.734006734006734
              precision    recall  f1-score   support

           0       0.66      0.98      0.79       297
           1       0.96      0.49      0.65       297

    accuracy                           0.73       594
   macro avg       0.81      0.73      0.72       594
weighted avg       0.81      0.73      0.72       594

Example prediction: Input:.. 

These seem hardly like the groups to discuss this in, but HUH??? 
All legitimate power to enforce these rights derives from the consent 
of the governed, not from no steenkin' piece of paper.  Civilized gov'mnt 
is not an autonomous computer program, it's interactive.  The Constitution 
was made by the people and can be trashed by us - it ain't no sacred 
scripture from which rights flow.  Our 'rights' come from our souls. 
And I sure didn't see any request to vote on trashing the sky.  
Again - my opinion only - we keep our rights by using them, not going to 
some court.  

 => Predicted: 1
